In [ ]:
import os
import json
import random
import shutil
import re
import math
import time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from pathlib import Path
import multiprocessing as mp
from matplotlib import pyplot as plt
from pathlib import Path
# load_dotenv(find_dotenv(usecwd=True))

HOME = "/home/michal/slrm/gen4"
if os.getenv("PLG_GROUPS_STORAGE"):
    HOME = "/net/people/plgrid/plgmichalgodek/workspace/ai-proton-simulations/gen4/"
os.chdir(HOME)


In [ ]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
training_data = np.load(Path(HOME,"training_data_g4_batch.npz"))
data_z_dose = training_data["data_z_dose"]
data_z_fluence_protons = training_data["data_z_fluence_protons"]
data_z_dlet_protons = training_data["data_z_dlet_protons"]
data_r_dose = training_data["data_r_dose"]
data_r_fluence_protons = training_data["data_r_fluence_protons"]
data_r_dlet_protons = training_data["data_r_dlet_protons"]
data_x = training_data["data_x"]

print(f"Training on energies [{data_x[0],data_x[1]}...{data_x[-2],data_x[-1]}")
x_min, x_max = np.min(data_x), np.max(data_x)

# per-series maxima
max_z_dose = np.max(data_z_dose)
max_r_dose = np.max(data_r_dose)
# overall max (kept for compatibility)

max_z_fluence_protons = np.max(data_z_fluence_protons)
max_r_fluence_protons = np.max(data_r_fluence_protons)

max_z_dlet_protons = np.max(data_z_dlet_protons)
max_r_dlet_protons = np.max(data_r_dlet_protons)

normalized_x = (data_x - x_min) / (x_max-x_min)

# normalized per-series
normalized_data_z_dose = data_z_dose / max_z_dose
normalized_data_r_dose = data_r_dose / max_r_dose

normalized_data_z_fluence_protons = data_z_fluence_protons / max_z_fluence_protons
normalized_data_r_fluence_protons = data_r_fluence_protons / max_r_fluence_protons

normalized_data_z_dlet_protons = data_z_dlet_protons / max_z_dlet_protons
normalized_data_r_dlet_protons = data_r_dlet_protons / max_r_dlet_protons

test_data = np.load(Path(HOME,"test_data_g4_batch5_test.npz"))
data_z_dose_test = test_data["data_z_dose_test"]
data_r_dose_test = test_data["data_r_dose_test"]
data_z_fluence_protons_test = test_data["data_z_fluence_protons_test"]
data_r_fluence_protons_test = test_data["data_r_fluence_protons_test"]
data_z_dlet_protons_test = test_data["data_z_dlet_protons_test"]
data_r_dlet_protons_test = test_data["data_r_dlet_protons_test"]
data_x_test = test_data["data_x_test"]

normalized_x_test = (data_x_test - x_min) / (x_max-x_min)
normalized_data_z_dose_test = data_z_dose_test / max_z_dose
normalized_data_r_dose_test = data_r_dose_test / max_r_dose
normalized_data_z_fluence_protons_test = data_z_fluence_protons_test / max_z_fluence_protons
normalized_data_r_fluence_protons_test = data_r_fluence_protons_test / max_r_fluence_protons
normalized_data_z_dlet_protons_test = data_z_dlet_protons_test / max_z_dlet_protons
normalized_data_r_dlet_protons_test = data_r_dlet_protons_test / max_r_dlet_protons

seeds_per_energy = 0
for i,x in enumerate(data_x):
    if float(x) == data_x[0]:
        seeds_per_energy += 1
    else:
        break

print(seeds_per_energy)


In [ ]:
import plotly.io as pio
pio.renderers.default = "notebook_connected" 


In [ ]:
plt.figure(figsize=(10,10))
for i in data_z_dose[::1]: 
    plt.plot(i)
plt.show()

plt.figure(figsize=(10,10))
for i in data_z_fluence_protons[::1]: 
    plt.plot(i)
plt.show()

plt.figure(figsize=(10,10))
for i in data_z_dlet_protons[::1]: 
    plt.plot(i)
plt.show()

plt.figure(figsize=(10,10))
for i in data_r_dose[::1]: 
    plt.plot(i)
plt.show()

plt.figure(figsize=(10,10))
for i in data_r_fluence_protons[::1]: 
    plt.plot(i)
plt.show()

plt.figure(figsize=(10,10))
for i in data_r_dlet_protons[::1]: 
    plt.plot(i)
plt.show()


In [ ]:

plt.figure(figsize=(10,10))
for i in data_z_dose_test[::1]: 
    plt.plot(i)
plt.show()

plt.figure(figsize=(10,10))
for i in data_z_fluence_protons_test[::1]: 
    plt.plot(i)
plt.show()

plt.figure(figsize=(10,10))
for i in data_z_dlet_protons_test[::1]: 
    plt.plot(i)
plt.show()

plt.figure(figsize=(10,10))
for i in data_r_dose_test[::1]: 
    plt.plot(i)
plt.show()

plt.figure(figsize=(10,10))
for i in data_r_fluence_protons_test[::1]: 
    plt.plot(i)
plt.show()

plt.figure(figsize=(10,10))
for i in data_r_dlet_protons_test[::1]: 
    plt.plot(i)
plt.show()


In [ ]:
# fig, ax = plt.subplots(figsize=(10,10))
# # plt.figure(figsize=(10,10))

# ax.set_yscale('log', base=10)
# for i in data_z_dose[::1]: 
#     ax.plot(i)
# fig.show()


# plt.figure(figsize=(10,10))
# for i in data_z_fluence_protons[::1]: 
#     plt.plot(i)
# plt.show()

# plt.figure(figsize=(10,10))
# for i in data_z_dlet_protons[::1]: 
#     plt.plot(i)
# plt.show()


In [ ]:
# import plotly.graph_objects as go
# fig = go.Figure()
# fig.update_layout(
#     width=900,   # px, default is 700
#     height=700,   # px, default is 450
#     autosize=False,  # must be False to fix the size
# )
# x = list(range(400))

# for i in range(1,31):
#     fig.add_trace(go.Scatter(x=x, y=data_z_dose[-i,:], mode="lines", showlegend=False, opacity = 1, line={"color": "red"}))
#     # print(data_x[-i])
# fig.add_trace(go.Scatter(x=x, y=data_z_dose_test[-1,:], mode="lines", showlegend=False, opacity = 1, line={"color": "blue"}))
# fig.show()


 # Load model

In [ ]:

class Model(nn.Module):
    def __init__(self, hidden_dim=128):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.n_z = 400
        self.n_r = 100

        self.trunk = nn.Sequential(
            nn.Linear(1, self.hidden_dim),
            nn.SiLU(),
            nn.Linear(self.hidden_dim, self.hidden_dim),
            nn.SiLU(),
            nn.Linear(self.hidden_dim, self.hidden_dim),
            nn.SiLU(),
        )

        # depth heads
        self.head_dose_z = nn.Sequential(
            nn.Linear(self.hidden_dim, self.n_z),
            nn.Softplus()
        )
        self.head_fluence_z = nn.Sequential(
            nn.Linear(self.hidden_dim, self.n_z),
            nn.Softplus()
        )
        self.head_dlet_z = nn.Sequential(
            nn.Linear(self.hidden_dim, self.n_z),
            nn.Softplus()
        )

        # lateral heads
        self.head_dose_r = nn.Linear(self.hidden_dim, self.n_r)
        self.head_fluence_r = nn.Linear(self.hidden_dim, self.n_r)
        self.head_dlet_r = nn.Sequential(
            nn.Linear(self.hidden_dim, self.n_r),
            nn.Softplus()
        )

    def forward(self, energy: torch.Tensor):
        if energy.dim() == 1:
            energy = energy.unsqueeze(-1)

        features = self.trunk(energy)

        dose_z = self.head_dose_z(features)
        fluence_z = self.head_fluence_z(features)
        dlet_z = self.head_dlet_z(features)

        dose_r = self.head_dose_r(features)
        fluence_r = self.head_fluence_r(features)
        dlet_r = self.head_dlet_r(features)

        return {
            "dose_z": dose_z,
            "fluence_z": fluence_z,
            "dlet_z": dlet_z,
            "dose_r": dose_r,
            "fluence_r": fluence_r,
            "dlet_r": dlet_r,
        }


In [ ]:
model = torch.load( HOME+'/checkpoints/basic_model.pth', weights_only=False,  map_location=device)#edit before run
# state_dict = torch.load( './checkpoints/gen3_batch2.pth',  map_location=device)
# model  = Model()
# model.load_state_dict(state_dict)
# model = model.to(device)
model = model.to(device)
model.eval()


 # Make predictions

In [ ]:
# print(normalized_x)


In [ ]:
model.eval()
X_norm_unique = np.unique(normalized_x).reshape(-1, 1)

with torch.no_grad():
    predictions = model(torch.from_numpy(X_norm_unique.astype(np.float32)).to(device))


In [ ]:
for key, value in predictions.items():
    print(key)
    for distribution in value:
        plt.plot(distribution.cpu().numpy())
        plt.title(key)
    predictions[key] = value.cpu().numpy()
    plt.show()


In [ ]:
# print(X_norm_unique)


In [ ]:
# X2 = X + 0.5
# # apply same min-max scaling to modified energies
# if X_max == X_min:
#     X2_norm = np.zeros_like(X2)
# else:
#     X2_norm = (X2 - X_min) / (X_max - X_min)
# X2_norm_unique = np.unique(X2_norm).reshape(-1, 1)


# with torch.no_grad():
#     predictions2 = model(torch.from_numpy(X2_norm_unique.astype(np.float32)).to(device)).cpu().numpy()


In [ ]:
plt.figure(figsize=(15,15))
# for i in predictions['dose_z']:
#     plt.plot(i)


In [ ]:
criterion = nn.HuberLoss(delta=0.08)


In [ ]:
# def test_model(model, criterion, device, batch_size=1):
#     """
#     Evaluates the trained model on the test dataset.
#     """
#     # 1. Set the model to evaluation mode
#     model.eval()
    
#     # 2. Prepare test tensors and move them to the correct device
#     Y_test = torch.stack([
#         torch.tensor(normalized_z_data_dose_test, dtype=torch.float32),
#         torch.tensor(normalized_data_z_fluence_protons_test, dtype=torch.float32),
#         torch.tensor(normalized_data_z_dlet_protons_test, dtype=torch.float32),
#     ], dim=1).to(device)
    
#     X_test_tensor = torch.tensor(data_x_test, dtype=torch.float32).to(device)
#     n_test = len(X_test_tensor)
    
#     total_test_loss = 0.0
    
#     # 3. Disable gradient computation to save memory and speed up inference
#     with torch.no_grad():
#         for i in range(0, n_test, batch_size):
#             x_batch = X_test_tensor[i:i+batch_size]
#             y_batch = Y_test[i:i+batch_size]
            
#             # Forward pass
#             pred = model(x_batch)
#             loss = criterion(pred, y_batch)
#             total_test_loss += loss.item()
            
#     # Calculate the average test loss across all batches
#     # total_test_loss = math.ceil(n_test / batch_size)
#     # avg_test_loss = test_loss / num_batches
    
#     return total_test_loss

# test_model(model, criterion, device)


 ## Compare against training data

 ## Plot predictions and true data

 ### Dose z

In [ ]:
import plotly.graph_objects as go
def prep_fig(title):
    fig = go.Figure()
    fig.update_layout(
        width=1200,   # px, default is 700
        height=1000,   # px, default is 450
        autosize=False,  # must be False to fix the size
        title = title,
    )
    return fig
x = list(range(400))

colors = ["red", "blue", "green", "orange"]
col_len = len(colors)

every_nth = 20
def plot_true_pred(true, pred, multiplier, quantity):

    metric = (pred-true)*multiplier

    fig = prep_fig(f"{quantity}:" + "for all energies" if every_nth == 1 else f"for every {every_nth}(th) energy" )
    for i, (tr,pr) in enumerate(zip(true[::every_nth],pred[::every_nth])):
        fig.add_trace(go.Scatter(x=x, y=pr*multiplier, mode="lines", showlegend=False, opacity = 1, line={"color": colors[i%col_len]}))
        fig.add_trace(go.Scatter(x=x, y=tr*multiplier, mode="lines", showlegend=False, opacity = 0.4, line={"color": colors[i%col_len]}))

    fig.show()
    
    fig = prep_fig(f"{quantity}: for lowest energy {data_x[0]} MeV")
    fig.add_trace(go.Scatter(x=x, y=pred[0]*multiplier, mode="lines", showlegend=False, opacity = 1, line={"color": colors[i%col_len]}))
    fig.add_trace(go.Scatter(x=x, y=true[0]*multiplier, mode="lines", showlegend=False, opacity = 0.4, line={"color": colors[i%col_len]}))


    fig.show()
    fig = prep_fig(f"{quantity}: for highest energy {data_x[-1]} MeV")
    fig.add_trace(go.Scatter(x=x, y=pred[-1]*multiplier, mode="lines", showlegend=False, opacity = 1, line={"color": colors[i%col_len]}))
    fig.add_trace(go.Scatter(x=x, y=true[-1]*multiplier, mode="lines", showlegend=False, opacity = 0.4, line={"color": colors[i%col_len]}))

    fig.show()    

plot_true_pred(normalized_data_z_dose[::seeds_per_energy], predictions['dose_z'], max_z_dose, "Dose z")



 ### Fluence z

In [ ]:
plot_true_pred(normalized_data_z_fluence_protons[::seeds_per_energy], predictions['fluence_z'], max_z_fluence_protons, "Fluence z")


 ### dLET z

In [ ]:
plot_true_pred(normalized_data_z_dlet_protons[::seeds_per_energy], predictions['dlet_z'], max_z_dlet_protons, "dLet z")


 ### Dose r

 ### Fluence r

In [ ]:
plot_true_pred(normalized_data_r_fluence_protons[::seeds_per_energy], predictions['fluence_r'], max_r_fluence_protons, "Fluence r")


 ### dLET r

In [ ]:
plot_true_pred(normalized_data_r_dlet_protons[::seeds_per_energy], predictions['dlet_r'], max_r_dlet_protons, "dLet r")



 ## Logarythmic scale

 ### Dose

In [ ]:
import plotly.graph_objects as go
def prep_fig(title):
    fig = go.Figure()
    fig.update_layout(
        width=1200,   # px, default is 700
        height=1000,   # px, default is 450
        autosize=False,  # must be False to fix the size
        title = title,
    )
    return fig
x = list(range(400))

colors = ["red", "blue", "green", "orange"]
col_len = len(colors)

every_nth = 10
def plot_true_pred_log(true, pred, multiplier, quantity):

    metric = (pred-true)*multiplier

    fig = prep_fig(f"{quantity}:  for all energies")
    for i, (tr,pr) in enumerate(zip(true,pred)):
        fig.add_trace(go.Scatter(x=x, y=pr*multiplier, mode="lines", showlegend=False, opacity = 1, line={"color": colors[i%col_len]}))
        fig.add_trace(go.Scatter(x=x, y=tr*multiplier, mode="lines", showlegend=False, opacity = 0.4, line={"color": colors[i%col_len]}))
    fig.update_yaxes(type="log")
    fig.show()
    
    fig = prep_fig(f"{quantity}: for lowest energy {data_x[0]} MeV")
    fig.add_trace(go.Scatter(x=x, y=pred[0]*multiplier, mode="lines", showlegend=False, opacity = 1, line={"color": colors[i%col_len]}))
    fig.add_trace(go.Scatter(x=x, y=true[0]*multiplier, mode="lines", showlegend=False, opacity = 0.4, line={"color": colors[i%col_len]}))
    fig.update_yaxes(type="log")
    fig.show()
    
    fig = prep_fig(f"{quantity}: for highest energy {data_x[-1]} MeV")
    fig.add_trace(go.Scatter(x=x, y=pred[-1]*multiplier, mode="lines", showlegend=False, opacity = 1, line={"color": colors[i%col_len]}))
    fig.add_trace(go.Scatter(x=x, y=true[-1]*multiplier, mode="lines", showlegend=False, opacity = 0.4, line={"color": colors[i%col_len]}))
    fig.update_yaxes(type="log")
    fig.show()    

plot_true_pred_log(normalized_data_z_dose[::seeds_per_energy], predictions['dose_z'], max_z_dose, "Dose z")



 ### Fluence z

In [ ]:
plot_true_pred_log(normalized_data_z_fluence_protons[::seeds_per_energy],  predictions['fluence_z'], max_z_fluence_protons, "Fluence z")


 ### dLet z

In [ ]:
plot_true_pred_log(normalized_data_z_dlet_protons[::seeds_per_energy],  predictions['dlet_z'], max_z_dlet_protons, "dLet z")


 ## difference

 ### Dose r

In [ ]:
plot_true_pred_log(normalized_data_r_dose[::seeds_per_energy], predictions['dose_r'], max_r_dose, "Dose r")

 ### Fluence r

In [ ]:
plot_true_pred_log(normalized_data_r_fluence_protons[::seeds_per_energy],  predictions['fluence_r'], max_r_fluence_protons, "Fluence r")


 ### dLet r

In [ ]:
plot_true_pred_log(normalized_data_r_dlet_protons[::seeds_per_energy],  predictions['dlet_r'], max_r_dlet_protons, "dLet r")




 ### dose difference

In [ ]:
def prep_fig(title):
    fig = go.Figure()
    fig.update_layout(
        width=1200,   # px, default is 700
        height=1000,   # px, default is 450
        autosize=False,  # must be False to fix the size
        title = title
    )
    return fig
x = list(range(400))

colors = ["red", "blue", "green", "orange"]
col_len = len(colors)

every_nth = 10
def plot_diff(true, pred, multiplier, quantity):

    metric = (pred-true)*multiplier

    fig = prep_fig(f"{quantity}: Difference for all energies")
    for i,serie in enumerate(metric):
        fig.add_trace(go.Scatter(x=x, y=serie, mode="lines", showlegend=False, opacity = 1, line={"color": colors[i%col_len]}))

    fig.show()
    
    fig = prep_fig(f"{quantity}: Difference for lowest energy")
    fig.add_trace(go.Scatter(x=x, y=metric[0], mode="lines", showlegend=False, opacity = 1, line={"color": colors[i%col_len]}))


    fig.show()
    fig = prep_fig(f"{quantity}: Difference for highest energy")
    fig.add_trace(go.Scatter(x=x, y=metric[-1], mode="lines", showlegend=False, opacity = 1, line={"color": colors[i%col_len]}))


    fig.show()    

plot_diff(normalized_data_z_dose[::seeds_per_energy], predictions['dose_z'], max_z_dose, "Dose z")


 ### Fluence z difference

In [ ]:
plot_diff(normalized_data_z_fluence_protons[::seeds_per_energy], predictions['fluence_z'], max_z_dlet_protons, "Fluence z")


 ### Dlet z difference

In [ ]:
plot_diff(normalized_data_z_dlet_protons[::seeds_per_energy], predictions['dlet_z'], max_z_dlet_protons, "DLet z")


 ### Dose r difference

In [ ]:
plot_diff(normalized_data_r_fluence_protons[::seeds_per_energy], predictions['fluence_r'], max_r_dlet_protons, "Fluence r")


 ### Fluence r difference

In [ ]:
plot_diff(normalized_data_r_fluence_protons[::seeds_per_energy], predictions['fluence_r'], max_r_dlet_protons, "Fluence r")


 ### DLet r difference

In [ ]:
plot_diff(normalized_data_r_dlet_protons[::seeds_per_energy], predictions['dlet_r'], max_r_dlet_protons, "Dlet r")



 ## Percent diff

 ### Dose z percent difference

In [ ]:
def print_percent_diff(true, pred, quantity):

    metric = (true/pred-1)*100

    fig = prep_fig(f"{quantity}: Percent difference for all energies")
    for i,serie in enumerate(metric):
        fig.add_trace(go.Scatter(x=x, y=serie, mode="lines", showlegend=False, opacity = 1, line={"color": colors[i%col_len]}))

    fig.show()
    
    fig = prep_fig(f"{quantity}: Percent difference for lowest energy")
    fig.add_trace(go.Scatter(x=x, y=metric[0], mode="lines", showlegend=False, opacity = 1, line={"color": colors[i%col_len]}))


    fig.show()
    fig = prep_fig(f"{quantity}: Percent difference for highest energy")
    fig.add_trace(go.Scatter(x=x, y=metric[-1], mode="lines", showlegend=False, opacity = 1, line={"color": colors[i%col_len]}))


    fig.show()    

print_percent_diff(normalized_data_z_dose[::seeds_per_energy], predictions['dose_z'], "Dose z")


 ### Fluence z percent difference

In [ ]:
print_percent_diff(normalized_data_z_fluence_protons[::seeds_per_energy], predictions['fluence_z'], "Fluence z")


 ### Dlet z percent difference

In [ ]:
print_percent_diff(normalized_data_z_dlet_protons[::seeds_per_energy], predictions['dlet_z'], "dLet z")


 ### Dose r percent difference

In [ ]:
print_percent_diff(normalized_data_r_dose[::seeds_per_energy], predictions['dose_r'], "Dose r")


 ### Fluence r percent difference

In [ ]:
print_percent_diff(normalized_data_r_fluence_protons[::seeds_per_energy], predictions['fluence_r'], "Fluence r")


 ### Dlet r percent difference

In [ ]:
print_percent_diff(normalized_data_r_dlet_protons[::seeds_per_energy], predictions['dlet_r'], "DLet r")


 # Compare against test data

In [ ]:
model.eval()

with torch.no_grad():
    test_predictions = model(torch.from_numpy(normalized_x_test.astype(np.float32)).to(device)).cpu().numpy()
# test_predictions = test_predictions[1::2]


test_predictions_wholes = test_predictions[::2]
test_predictions_halves = test_predictions[1::2]

normalized_test_x_wholes, normalized_test_x_halves = normalized_x_test[::2], normalized_x_test[1::2]


normalized_data_z_dose_test_wholes, normalized_data_z_dose_test_halves = normalized_data_z_dose_test[::2], normalized_data_z_dose_test[1::2] 
normalized_data_z_fluence_protons_test_wholes, normalized_data_z_fluence_protons_test_halves = normalized_data_z_fluence_protons_test[::2], normalized_data_z_fluence_protons_test[1::2] 
normalized_data_z_dlet_protons_test_wholes, normalized_data_z_dlet_protons_test_halves = normalized_data_z_dlet_protons_test[::2], normalized_data_z_dlet_protons_test[1::2] 

normalized_data_r_dose_test_wholes, normalized_data_r_dose_test_halves = normalized_data_r_dose_test[::2], normalized_data_r_dose_test[1::2] 
normalized_data_r_fluence_protons_test_wholes, normalized_data_r_fluence_protons_test_halves = normalized_data_r_fluence_protons_test[::2], normalized_data_r_fluence_protons_test[1::2] 
normalized_data_r_dlet_protons_test_wholes, normalized_data_r_dlet_protons_test_halves = normalized_data_r_dlet_protons_test[::2], normalized_data_r_dlet_protons_test[1::2] 

In [ ]:
print(test_predictions.shape)


In [ ]:
plt.figure(figsize=(10,10))
plt.plot(predictions[0,0])
plt.plot(predictions[1,0])
plt.plot(test_predictions[1,0])


In [ ]:
test_predictions.shape

 ### Dose z

In [ ]:
import plotly.graph_objects as go
def prep_fig(title):
    fig = go.Figure()
    fig.update_layout(
        width=1200,   # px, default is 700
        height=1000,   # px, default is 450
        autosize=False,  # must be False to fix the size
        title = title,
    )
    return fig
x = list(range(400))


colors = ["red", "blue", "green", "orange"]
col_len = len(colors)


every_nth = 10
def plot_true_pred(true, pred, multiplier, quantity):


    metric = (pred-true)*multiplier


    fig = prep_fig(f"{quantity}:  for all energies")
    for i, (tr,pr) in enumerate(zip(true[::20],pred[::20])):
        fig.add_trace(go.Scatter(x=x, y=pr*multiplier, mode="lines", showlegend=False, opacity = 1, line={"color": colors[i%col_len]}))
        fig.add_trace(go.Scatter(x=x, y=tr*multiplier, mode="lines", showlegend=False, opacity = 0.4, line={"color": colors[i%col_len]}))


    fig.show()
    
    fig = prep_fig(f"{quantity}: for lowest energy {data_x_test[1]} MeV")
    fig.add_trace(go.Scatter(x=x, y=pred[0]*multiplier, mode="lines", showlegend=False, opacity = 1, line={"color": colors[i%col_len]}))
    fig.add_trace(go.Scatter(x=x, y=true[0]*multiplier, mode="lines", showlegend=False, opacity = 0.4, line={"color": colors[i%col_len]}))



    fig.show()
    fig = prep_fig(f"{quantity}: for highest energy {data_x_test[-1]} MeV")
    fig.add_trace(go.Scatter(x=x, y=pred[-1]*multiplier, mode="lines", showlegend=False, opacity = 1, line={"color": colors[i%col_len]}))
    fig.add_trace(go.Scatter(x=x, y=true[-1]*multiplier, mode="lines", showlegend=False, opacity = 0.4, line={"color": colors[i%col_len]}))



    fig.show()    


plot_true_pred(normalized_data_z_dose_test_wholes, test_predictions_wholes['dose_z'], max_z_dose, "Dose z")




 ### Fluence z

In [ ]:
plot_true_pred(normalized_data_z_fluence_protons_test_wholes[::], test_predictions_wholes['fluence_z'], max_z_fluence_protons, "Fluence z")



 ### dLET z

In [ ]:
plot_true_pred(normalized_data_z_dlet_protons_test_wholes[::], test_predictions_wholes['dlet_z'], max_z_dlet_protons, "dLet z")



 ### Dose r

In [ ]:
plot_true_pred(normalized_data_r_dose_test_wholes, test_predictions_wholes['dose_r'], max_r_dose, "Dose r")



 ### Fluence r

In [ ]:
plot_true_pred(normalized_data_r_fluence_protons_test_wholes[::], test_predictions_wholes['fluence_r'], max_r_fluence_protons, "Fluence r")



 ### dLET r

In [ ]:
plot_true_pred(normalized_data_r_dlet_protons_test_wholes[::], test_predictions_wholes['dlet_r'], max_r_dlet_protons, "dLet r")




 ## Logarythmic scale

 ### Dose z

In [ ]:
import plotly.graph_objects as go
def prep_fig(title):
    fig = go.Figure()
    fig.update_layout(
        width=1200,   # px, default is 700
        height=1000,   # px, default is 450
        autosize=False,  # must be False to fix the size
        title = title,
    )
    return fig
x = list(range(400))


colors = ["red", "blue", "green", "orange"]
col_len = len(colors)


every_nth = 10
def plot_true_pred_log(true, pred, multiplier, quantity):


    metric = (pred-true)*multiplier


    fig = prep_fig(f"{quantity}:  for all energies")
    for i, (tr,pr) in enumerate(zip(true[::20],pred[::20])):
        fig.add_trace(go.Scatter(x=x, y=pr*multiplier, mode="lines", showlegend=False, opacity = 1, line={"color": colors[i%col_len]}))
        fig.add_trace(go.Scatter(x=x, y=tr*multiplier, mode="lines", showlegend=False, opacity = 0.4, line={"color": colors[i%col_len]}))
    fig.update_yaxes(type="log")
    fig.show()
    
    fig = prep_fig(f"{quantity}: for lowest energy {data_x[0]} MeV")
    fig.add_trace(go.Scatter(x=x, y=pred[0]*multiplier, mode="lines", showlegend=False, opacity = 1, line={"color": colors[i%col_len]}))
    fig.add_trace(go.Scatter(x=x, y=true[0]*multiplier, mode="lines", showlegend=False, opacity = 0.4, line={"color": colors[i%col_len]}))
    fig.update_yaxes(type="log")
    fig.show()
    
    fig = prep_fig(f"{quantity}: for highest energy {data_x[-1]} MeV")
    fig.add_trace(go.Scatter(x=x, y=pred[-1]*multiplier, mode="lines", showlegend=False, opacity = 1, line={"color": colors[i%col_len]}))
    fig.add_trace(go.Scatter(x=x, y=true[-1]*multiplier, mode="lines", showlegend=False, opacity = 0.4, line={"color": colors[i%col_len]}))
    fig.update_yaxes(type="log")
    fig.show()    


plot_true_pred_log(normalized_data_z_dose_test_wholes[::], test_predictions_wholes['dose_z'], max_z_dose, "Dose z")




 ### Fluence z

In [ ]:
plot_true_pred_log(normalized_data_z_fluence_protons_test_wholes[::],  test_predictions_wholes['fluence_z'], max_z_fluence_protons, "Fluence z")



 ### dLet z

In [ ]:
plot_true_pred_log(normalized_data_z_dlet_protons_test_wholes[::],  test_predictions_wholes['dlet_z'], max_z_dlet_protons, "dLet z")



 ### Dose r

In [ ]:
plot_true_pred_log(normalized_data_r_dose_test_wholes[::], test_predictions_wholes['dose_r'], max_r_dose, "Dose r")



 ### Fluence r

In [ ]:
plot_true_pred_log(normalized_data_r_fluence_protons_test_wholes[::],  test_predictions_wholes['fluence_r'], max_r_fluence_protons, "Fluence r")



 ### dLet r

In [ ]:
plot_true_pred_log(normalized_data_r_dlet_protons_test_wholes[::],  test_predictions_wholes['dlet_r'], max_r_dlet_protons, "dLet r")



 ## difference

 ### dose z difference

In [ ]:
def prep_fig(title):
    fig = go.Figure()
    fig.update_layout(
        width=1200,   # px, default is 700
        height=1000,   # px, default is 450
        autosize=False,  # must be False to fix the size
        title = title,
    )
    return fig
x = list(range(400))


colors = ["red", "blue", "green", "orange"]
col_len = len(colors)


every_nth = 10
def plot_diff(true, pred, multiplier, quantity):


    metric = (pred-true)*multiplier


    fig = prep_fig(f"{quantity}: Difference for all energies")
    for i,serie in enumerate(metric):
        fig.add_trace(go.Scatter(x=x, y=serie, mode="lines", showlegend=False, opacity = 1, line={"color": colors[i%col_len]}))


    fig.show()
    
    fig = prep_fig(f"{quantity}: Difference for lowest energy")
    fig.add_trace(go.Scatter(x=x, y=metric[0], mode="lines", showlegend=False, opacity = 1, line={"color": colors[i%col_len]}))



    fig.show()
    fig = prep_fig(f"{quantity}: Difference for highest energy")
    fig.add_trace(go.Scatter(x=x, y=metric[-1], mode="lines", showlegend=False, opacity = 1, line={"color": colors[i%col_len]}))



    fig.show()    


plot_diff(normalized_data_z_dose_test_wholes[::], test_predictions_wholes['dose_z'], max_z_dose, "Dose z")



 ### fluence z difference

In [ ]:
plot_diff(normalized_data_z_fluence_protons_test_wholes[::], test_predictions_wholes['fluence_z'], max_z_dlet_protons, "Fluence z")



 ### dlet z difference

In [ ]:
plot_diff(normalized_data_z_dlet_protons_test_wholes[::], test_predictions_wholes['dlet_z'], max_z_dlet_protons, "dlet z")



 ### dose r difference

In [ ]:
plot_diff(normalized_data_r_dose_test_wholes[::], test_predictions_wholes['dose_r'], max_r_dose, "Dose r")



 ### fluence r difference

In [ ]:
plot_diff(normalized_data_r_fluence_protons_test_wholes[::], test_predictions_wholes['fluence_r'], max_r_dlet_protons, "Fluence r")



 ### dlet r difference

In [ ]:
plot_diff(normalized_data_r_dlet_protons_test_wholes[::], test_predictions_wholes['dlet_r'], max_r_dlet_protons, "dlet r")



 ## Percent diff

 ### Dose z percent difference

In [ ]:
def print_percent_diff(true, pred, quantity):


    metric = (true/pred-1)*100


    fig = prep_fig(f"{quantity}: Percent difference for all energies")
    for i,serie in enumerate(metric[::]):
        fig.add_trace(go.Scatter(x=x, y=serie, mode="lines", showlegend=False, opacity = 1, line={"color": colors[i%col_len]}))


    fig.show()
    
    fig = prep_fig(f"{quantity}: Percent difference for lowest energy")
    fig.add_trace(go.Scatter(x=x, y=metric[0], mode="lines", showlegend=False, opacity = 1, line={"color": colors[i%col_len]}))



    fig.show()
    fig = prep_fig(f"{quantity}: Percent difference for highest energy")
    fig.add_trace(go.Scatter(x=x, y=metric[-1], mode="lines", showlegend=False, opacity = 1, line={"color": colors[i%col_len]}))



    fig.show()    


print_percent_diff(normalized_data_z_dose_test_wholes[::], test_predictions_wholes['dose_z'], "Dose z")



 ### Fluence z percent difference

In [ ]:
print_percent_diff(normalized_data_z_fluence_protons_test_wholes[::], test_predictions_wholes['fluence_z'], "Fluence z")



 ### Dlet z percent difference

In [ ]:
print_percent_diff(normalized_data_z_dlet_protons_test_wholes[::], test_predictions_wholes['dlet_z'], "dLet z")



 ### Dose r percent difference

In [ ]:
print_percent_diff(normalized_data_r_dose_test_wholes[::], test_predictions_wholes['dose_r'], "Dose r")



 ### Fluence r percent difference

In [ ]:
print_percent_diff(normalized_data_r_fluence_protons_test_wholes[::], test_predictions_wholes['fluence_r'], "Fluence r")



 ### Dlet r percent difference

In [ ]:
print_percent_diff(normalized_data_r_dlet_protons_test_wholes[::], test_predictions_wholes['dlet_r'], "dLet r")



 # Compare against test halves

 # Compare against test halves

In [ ]:
plt.figure(figsize=(10,10))
plt.plot(predictions[0,0], color="blue")
plt.plot(predictions[1,0], color="red")
plt.plot(test_predictions[1,0], color="yellow")



In [ ]:
test_predictions.shape



 ### Dose z

In [ ]:
import plotly.graph_objects as go
def prep_fig(title):
    fig = go.Figure()
    fig.update_layout(
        width=1200,   # px, default is 700
        height=1000,   # px, default is 450
        autosize=False,  # must be False to fix the size
        title = title,
    )
    return fig
x = list(range(400))


colors = ["red", "blue", "green", "orange"]
col_len = len(colors)


every_nth = 10
def plot_true_pred(true, pred, multiplier, quantity):


    metric = (pred-true)*multiplier


    fig = prep_fig(f"{quantity}:  for all energies")
    for i, (tr,pr) in enumerate(zip(true[::20],pred[::20])):
        fig.add_trace(go.Scatter(x=x, y=pr*multiplier, mode="lines", showlegend=False, opacity = 1, line={"color": colors[i%col_len]}))
        fig.add_trace(go.Scatter(x=x, y=tr*multiplier, mode="lines", showlegend=False, opacity = 0.4, line={"color": colors[i%col_len]}))


    fig.show()
    
    fig = prep_fig(f"{quantity}: for lowest energy {data_x_test[1]} MeV")
    fig.add_trace(go.Scatter(x=x, y=pred[0]*multiplier, mode="lines", showlegend=False, opacity = 1, line={"color": colors[i%col_len]}))
    fig.add_trace(go.Scatter(x=x, y=true[0]*multiplier, mode="lines", showlegend=False, opacity = 0.4, line={"color": colors[i%col_len]}))



    fig.show()
    fig = prep_fig(f"{quantity}: for highest energy {data_x_test[-1]} MeV")
    fig.add_trace(go.Scatter(x=x, y=pred[-1]*multiplier, mode="lines", showlegend=False, opacity = 1, line={"color": colors[i%col_len]}))
    fig.add_trace(go.Scatter(x=x, y=true[-1]*multiplier, mode="lines", showlegend=False, opacity = 0.4, line={"color": colors[i%col_len]}))



    fig.show()    


plot_true_pred(normalized_data_z_dose_test_halves, test_predictions_halves['dose_z'], max_z_dose, "Dose z")




 ### Fluence z

In [ ]:
plot_true_pred(normalized_data_z_fluence_protons_test_halves[::], test_predictions_halves['fluence_z'], max_z_fluence_protons, "Fluence z")



 ### dLET z

In [ ]:
plot_true_pred(normalized_data_z_dlet_protons_test_halves[::], test_predictions_halves['dlet_z'], max_z_dlet_protons, "dLet z")



 ### Dose r

In [ ]:
plot_true_pred(normalized_data_r_dose_test_halves, test_predictions_halves['dose_r'], max_r_dose, "Dose r")



 ### Fluence r

In [ ]:
plot_true_pred(normalized_data_r_fluence_protons_test_halves[::], test_predictions_halves['fluence_r'], max_r_fluence_protons, "Fluence r")



 ### dLET r

In [ ]:
plot_true_pred(normalized_data_r_dlet_protons_test_halves[::], test_predictions_halves['dlet_r'], max_r_dlet_protons, "dLet r")




 ## Logarythmic scale

 ### Dose z

In [ ]:
import plotly.graph_objects as go
def prep_fig(title):
    fig = go.Figure()
    fig.update_layout(
        width=1200,   # px, default is 700
        height=1000,   # px, default is 450
        autosize=False,  # must be False to fix the size
        title = title,
    )
    return fig
x = list(range(400))


colors = ["red", "blue", "green", "orange"]
col_len = len(colors)


every_nth = 10
def plot_true_pred_log(true, pred, multiplier, quantity):


    metric = (pred-true)*multiplier


    fig = prep_fig(f"{quantity}:  for all energies")
    for i, (tr,pr) in enumerate(zip(true[::20],pred[::20])):
        fig.add_trace(go.Scatter(x=x, y=pr*multiplier, mode="lines", showlegend=False, opacity = 1, line={"color": colors[i%col_len]}))
        fig.add_trace(go.Scatter(x=x, y=tr*multiplier, mode="lines", showlegend=False, opacity = 0.4, line={"color": colors[i%col_len]}))
    fig.update_yaxes(type="log")
    fig.show()
    
    fig = prep_fig(f"{quantity}: for lowest energy {data_x[0]} MeV")
    fig.add_trace(go.Scatter(x=x, y=pred[0]*multiplier, mode="lines", showlegend=False, opacity = 1, line={"color": colors[i%col_len]}))
    fig.add_trace(go.Scatter(x=x, y=true[0]*multiplier, mode="lines", showlegend=False, opacity = 0.4, line={"color": colors[i%col_len]}))
    fig.update_yaxes(type="log")
    fig.show()
    
    fig = prep_fig(f"{quantity}: for highest energy {data_x[-1]} MeV")
    fig.add_trace(go.Scatter(x=x, y=pred[-1]*multiplier, mode="lines", showlegend=False, opacity = 1, line={"color": colors[i%col_len]}))
    fig.add_trace(go.Scatter(x=x, y=true[-1]*multiplier, mode="lines", showlegend=False, opacity = 0.4, line={"color": colors[i%col_len]}))
    fig.update_yaxes(type="log")
    fig.show()    


plot_true_pred_log(normalized_data_z_dose_test_halves[::], test_predictions_halves['dose_z'], max_z_dose, "Dose z")




 ### Fluence z

In [ ]:
plot_true_pred_log(normalized_data_z_fluence_protons_test_halves[::],  test_predictions_halves['fluence_z'], max_z_fluence_protons, "Fluence z")



 ### dLet z

In [ ]:
plot_true_pred_log(normalized_data_z_dlet_protons_test_halves[::],  test_predictions_halves['dlet_z'], max_z_dlet_protons, "dLet z")



 ### Dose r

In [ ]:
plot_true_pred_log(normalized_data_r_dose_test_halves[::], test_predictions_halves['dose_r'], max_r_dose, "Dose r")



 ### Fluence r

In [ ]:
plot_true_pred_log(normalized_data_r_fluence_protons_test_halves[::],  test_predictions_halves['fluence_r'], max_r_fluence_protons, "Fluence r")



 ### dLet r

In [ ]:
plot_true_pred_log(normalized_data_r_dlet_protons_test_halves[::],  test_predictions_halves['dlet_r'], max_r_dlet_protons, "dLet r")



 ## difference

 ### dose z difference

In [ ]:
def prep_fig(title):
    fig = go.Figure()
    fig.update_layout(
        width=1200,   # px, default is 700
        height=1000,   # px, default is 450
        autosize=False,  # must be False to fix the size
        title = title,
    )
    return fig
x = list(range(400))


colors = ["red", "blue", "green", "orange"]
col_len = len(colors)


every_nth = 10
def plot_diff(true, pred, multiplier, quantity):


    metric = (pred-true)*multiplier


    fig = prep_fig(f"{quantity}: Difference for all energies")
    for i,serie in enumerate(metric):
        fig.add_trace(go.Scatter(x=x, y=serie, mode="lines", showlegend=False, opacity = 1, line={"color": colors[i%col_len]}))


    fig.show()
    
    fig = prep_fig(f"{quantity}: Difference for lowest energy")
    fig.add_trace(go.Scatter(x=x, y=metric[0], mode="lines", showlegend=False, opacity = 1, line={"color": colors[i%col_len]}))



    fig.show()
    fig = prep_fig(f"{quantity}: Difference for highest energy")
    fig.add_trace(go.Scatter(x=x, y=metric[-1], mode="lines", showlegend=False, opacity = 1, line={"color": colors[i%col_len]}))



    fig.show()    


plot_diff(normalized_data_z_dose_test_halves[::], test_predictions_halves['dose_z'], max_z_dose, "Dose z")



 ### fluence z difference

In [ ]:
plot_diff(normalized_data_z_fluence_protons_test_halves[::], test_predictions_halves['fluence_z'], max_z_dlet_protons, "Fluence z")



 ### dlet z difference

In [ ]:
plot_diff(normalized_data_z_dlet_protons_test_halves[::], test_predictions_halves['dlet_z'], max_z_dlet_protons, "dlet z")



 ### dose r difference

In [ ]:
plot_diff(normalized_data_r_dose_test_halves[::], test_predictions_halves['dose_r'], max_r_dose, "Dose r")



 ### fluence r difference

In [ ]:
plot_diff(normalized_data_r_fluence_protons_test_halves[::], test_predictions_halves['fluence_r'], max_r_dlet_protons, "Fluence r")



 ### dlet r difference

In [ ]:
plot_diff(normalized_data_r_dlet_protons_test_halves[::], test_predictions_halves['dlet_r'], max_r_dlet_protons, "dlet r")



 ## Percent diff

 ### Dose z percent difference

In [ ]:
def print_percent_diff(true, pred, quantity):


    metric = (true/pred-1)*100


    fig = prep_fig(f"{quantity}: Percent difference for all energies")
    for i,serie in enumerate(metric[::]):
        fig.add_trace(go.Scatter(x=x, y=serie, mode="lines", showlegend=False, opacity = 1, line={"color": colors[i%col_len]}))


    fig.show()
    
    fig = prep_fig(f"{quantity}: Percent difference for lowest energy")
    fig.add_trace(go.Scatter(x=x, y=metric[0], mode="lines", showlegend=False, opacity = 1, line={"color": colors[i%col_len]}))



    fig.show()
    fig = prep_fig(f"{quantity}: Percent difference for highest energy")
    fig.add_trace(go.Scatter(x=x, y=metric[-1], mode="lines", showlegend=False, opacity = 1, line={"color": colors[i%col_len]}))



    fig.show()    


print_percent_diff(normalized_data_z_dose_test_halves[::], test_predictions_halves['dose_z'], "Dose z")



 ### Fluence z percent difference

In [ ]:
print_percent_diff(normalized_data_z_fluence_protons_test_halves[::], test_predictions_halves['fluence_z'], "Fluence z")



 ### Dlet z percent difference

In [ ]:
print_percent_diff(normalized_data_z_dlet_protons_test_halves[::], test_predictions_halves['dlet_z'], "dLet z")



 ### Dose r percent difference

In [ ]:
print_percent_diff(normalized_data_r_dose_test_halves[::], test_predictions_halves['dose_r'], "Dose r")



 ### Fluence r percent difference

In [ ]:
print_percent_diff(normalized_data_r_fluence_protons_test_halves[::], test_predictions_halves['fluence_r'], "Fluence r")



 ### Dlet r percent difference

In [ ]:
print_percent_diff(normalized_data_r_dlet_protons_test_halves[::], test_predictions_halves['dlet_r'], "dLet r")